In [1]:
# CELL 1 — Install dependencies + mount Drive
import os
import sys
import subprocess as _sp
import shutil
from pathlib import Path


def _run(cmd, label):
    print(f"\n[{label}] {' '.join(map(str, cmd))}")
    result = _sp.run(cmd, text=True, capture_output=True)
    if result.returncode != 0:
        print(result.stdout[-2000:])
        print(result.stderr[-4000:])
        raise RuntimeError(f"{label} failed with exit code {result.returncode}")
    return result


def _pip_install(package):
    # Install packages separately so one resolver conflict does not hide which
    # dependency failed. --prefer-binary avoids unnecessary source builds.
    print(f"Installing {package} ...")
    result = _sp.run(
        [sys.executable, "-m", "pip", "install", "--prefer-binary", "--no-cache-dir", package],
        text=True, capture_output=True
    )
    if result.returncode != 0:
        print(result.stdout[-2000:])
        print(result.stderr[-5000:])
        raise RuntimeError(f"pip install {package!r} failed with exit code {result.returncode}")
    print(f"  OK: {package}")


print(f"Python: {sys.version.split()[0]}")
print("Installing Python dependencies...")

# Install independently rather than as one large pip transaction. This is
# substantially more robust in Colab when the base image already contains
# packages with pinned versions.
for _pkg in [
    "transformers",
    "accelerate",
    "bitsandbytes",
    "flawfinder",
    "bandit",
    "semgrep",
]:
    _pip_install(_pkg)

print("\nInstalling native/compiler dependencies...")
_run(["apt-get", "update", "-qq"], "apt update")
_run(
    ["apt-get", "install", "-y", "-qq",
     "gcc", "g++", "openjdk-17-jdk-headless", "unzip", "curl", "golang-go"],
    "compiler/JDK installation",
)

# SpotBugs is distributed as a standalone Java archive rather than a normal
# Python package. Install the official release and expose its CLI on PATH.
SPOTBUGS_VERSION = "4.8.6"
SPOTBUGS_ROOT = Path("/opt/spotbugs")
SPOTBUGS_BIN = SPOTBUGS_ROOT / "bin" / "spotbugs"

if not SPOTBUGS_BIN.exists():
    import urllib.request
    import zipfile
    import shutil

    _zip = Path(f"/tmp/spotbugs-{SPOTBUGS_VERSION}.zip")
    _url = (
        f"https://github.com/spotbugs/spotbugs/releases/download/"
        f"{SPOTBUGS_VERSION}/spotbugs-{SPOTBUGS_VERSION}.zip"
    )
    print(f"Downloading SpotBugs {SPOTBUGS_VERSION} ...")
    urllib.request.urlretrieve(_url, _zip)
    if SPOTBUGS_ROOT.exists():
        shutil.rmtree(SPOTBUGS_ROOT)
    _extract_root = Path("/tmp/spotbugs_extract")
    if _extract_root.exists():
        shutil.rmtree(_extract_root)
    _extract_root.mkdir(parents=True)
    with zipfile.ZipFile(_zip) as zf:
        zf.extractall(_extract_root)
    _dirs = [d for d in _extract_root.iterdir() if d.is_dir()]
    if len(_dirs) != 1:
        raise RuntimeError(f"Unexpected SpotBugs archive layout: {_dirs}")
    shutil.move(str(_dirs[0]), str(SPOTBUGS_ROOT))
    _zip.unlink(missing_ok=True)

os.environ["SPOTBUGS_HOME"] = str(SPOTBUGS_ROOT)
os.environ["PATH"] = str(SPOTBUGS_ROOT / "bin") + os.pathsep + os.environ.get("PATH", "")

print(f"SpotBugs: {shutil.which('spotbugs')}")
# Enry is installed as its official CLI, not via the unrelated/old PyPI
# source package that may try to build from setup.py on some runtimes.
ENRY_VERSION = "v1.3.0"
ENRY_BIN = Path("/root/go/bin/enry")
if not ENRY_BIN.exists():
    print(f"Installing Enry CLI {ENRY_VERSION} ...")
    _run(["go", "install", f"github.com/go-enry/enry@{ENRY_VERSION}"], "Enry installation")
os.environ["PATH"] = str(ENRY_BIN.parent) + os.pathsep + os.environ.get("PATH", "")
print(f"Enry CLI: {shutil.which('enry')}")
if not shutil.which("enry"):
    raise RuntimeError("Enry CLI installation completed but 'enry' is not on PATH.")
print(f"Bandit: {shutil.which('bandit')}")
print(f"Semgrep: {shutil.which('semgrep')}")

print("\nMounting Google Drive...")
from google.colab import drive
drive.mount("/content/drive")
print("All dependencies ready.")


Python: 3.13.15
Installing Python dependencies...
Installing transformers ...
  OK: transformers
Installing accelerate ...
  OK: accelerate
Installing bitsandbytes ...
  OK: bitsandbytes
Installing flawfinder ...
  OK: flawfinder
Installing bandit ...
  OK: bandit
Installing semgrep ...
  OK: semgrep

Installing native/compiler dependencies...

[apt update] apt-get update -qq

[compiler/JDK installation] apt-get install -y -qq gcc g++ openjdk-17-jdk-headless unzip curl golang-go
SpotBugs: None
Installing Enry CLI v1.3.0 ...

[Enry installation] go install github.com/go-enry/enry@v1.3.0
Enry CLI: /root/go/bin/enry
Bandit: /usr/local/bin/bandit
Semgrep: /usr/local/bin/semgrep

Mounting Google Drive...
Mounted at /content/drive
All dependencies ready.


In [2]:
import importlib.metadata as md
for p in ["transformers", "accelerate", "bitsandbytes", "torch",
          "semgrep", "bandit", "requests", "huggingface-hub", "tokenizers"]:
    try:
      print(f"{p:18} {md.version(p)}")
    except md.PackageNotFoundError:
      print(f"{p:18} NOT INSTALLED")

import subprocess
for c in (["flawfinder","--version"], ["gcc","--version"], ["g++","--version"],
          ["java","-version"], ["go","version"]):
    try:
      print(c[0], "->", subprocess.run(c, capture_output=True, text=True).stdout.splitlines()[0] or subprocess.run(c, capture_output=True,text=True).stderr.splitlines()[0])
    except Exception as e:
      print(c[0], "-> unavailable:", e)

transformers       5.16.1
accelerate         1.14.0
bitsandbytes       0.50.2
torch              2.11.0+cpu
semgrep            1.176.1
bandit             1.9.4
requests           2.32.4
huggingface-hub    1.28.0
tokenizers         0.23.1
flawfinder -> 2.0.20
gcc -> gcc (Ubuntu 11.4.0-1ubuntu1~22.04.3) 11.4.0
g++ -> g++ (Ubuntu 11.4.0-1ubuntu1~22.04.3) 11.4.0
java -> [0.027s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
go -> go version go1.18.1 linux/amd64


In [ ]:
# CELL 2 — Imports & paths
import os
import re
import json
import time
import shutil
import subprocess
import tempfile
from pathlib import Path
from datetime import datetime, timezone

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

if not shutil.which("flawfinder"):
    subprocess.run([sys.executable, "-m", "pip", "install", "flawfinder", "-q"], check=True)
    print("flawfinder installed.")

# ── CONFIGURE THESE (adjust if your Drive layout differs) ──────────────────
REPO_ROOT   = Path("/content/drive/MyDrive/")

HARNESS_DIR = REPO_ROOT / "testcasesupport" # io.c + std_testcase.h
INPUT_DIR   = REPO_ROOT / "L1_handoff"                 # non-CFR codes from L0 (read-only)
OUTPUT_DIR  = REPO_ROOT / "l1_outputs"                 # per-case L1 fixed files
RESULTS_DIR = REPO_ROOT / "l1_results"                 # L1 JSON results + progress
# ───────────────────────────────────────────────────────────────────────────

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

if not HARNESS_DIR.exists():
    raise FileNotFoundError(
        f"\n[ERROR] HARNESS_DIR not found: {HARNESS_DIR}\n"
        "Upload datasets/C/testcasesupport/ to your Drive under cve_llm/\n"
        "Or update HARNESS_DIR to the correct path."
    )
if not (HARNESS_DIR / "std_testcase.h").exists():
    raise FileNotFoundError(
        f"[ERROR] std_testcase.h not found in {HARNESS_DIR}\n"
        "Make sure the full testcasesupport/ folder is uploaded."
    )
print(f"  Harness files OK: {list(HARNESS_DIR.iterdir())}")

MODEL_ID = "Qwen/Qwen2.5-14B-Instruct"

SAST_ONLY_CWES = {
    "510",  # Trapdoor (logic-based)
    "367",  # TOC/TOU (race condition)
    "36",   # Absolute Path Traversal
    "78",   # OS Command Injection
}
NUM_HARNESS_RUNS = 3

print("Paths configured.")
print(f"  HARNESS_DIR    : {HARNESS_DIR}")
print(f"  INPUT_DIR      : {INPUT_DIR}")
print(f"  OUTPUT_DIR     : {OUTPUT_DIR}")
print(f"  RESULTS_DIR    : {RESULTS_DIR}")

In [ ]:
# CELL 3 — Load Qwen2.5-14B-Instruct (4-bit quantised for T4/A100)
from transformers import BitsAndBytesConfig

print(f"\nLoading {MODEL_ID} ...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map="auto",
    torch_dtype=torch.float16,
    quantization_config=quantization_config,
    trust_remote_code=True,
)
model.eval()
print("Model loaded.")

In [ ]:
# CELL 4 — Helper functions

def extract_cwe_id(dirname: str) -> str:
    """Extract CWE number from dir name, e.g. 'CWE121_Stack_...' -> '121'."""
    m = re.match(r"CWE(\d+)", dirname)
    return m.group(1) if m else ""


_WINDOWS_ONLY_PATTERNS = ["w32", "_winsock", "_win32"]


def _is_windows_only(src_file: Path) -> bool:
    stem_lower = src_file.stem.lower()
    return any(pat in stem_lower for pat in _WINDOWS_ONLY_PATTERNS)


print("Helper functions defined.")

In [ ]:
# CELL 5 — Load cases from L1_handoff

if not INPUT_DIR.exists():
    raise FileNotFoundError(f"[ERROR] INPUT_DIR not found: {INPUT_DIR}")

cases = []

for cwe_dir in sorted(INPUT_DIR.iterdir()):
    if not cwe_dir.is_dir():
        continue

    cwe_id = extract_cwe_id(cwe_dir.name)

    source_files = sorted(
        list(cwe_dir.glob("*.c")) +
        list(cwe_dir.glob("*.cpp")) +
        list(cwe_dir.glob("*.cc")) +
        list(cwe_dir.glob("*.cxx"))
    )

    for src_file in source_files:

        if _is_windows_only(src_file):
            continue

        case_id = f"{cwe_id}_{src_file.stem}"

        cases.append({
            "case_id":  case_id,
            "cwe_dir":  cwe_dir.name,
            "cwe_id":   cwe_id,
            "filepath": str(src_file),
            "filename": src_file.name,
            "source":   src_file.read_text(
                encoding="utf-8",
                errors="replace"
            ),
        })

print(f"L1 cases to process: {len(cases)}")
print(f"Input directory: {INPUT_DIR}")

print("\nCases by CWE:")
for cwe in sorted(set(c["cwe_dir"] for c in cases)):
    cwe_cases = [c for c in cases if c["cwe_dir"] == cwe]
    print(f"  {cwe}: {len(cwe_cases)} cases")

print("\nCase IDs:")
for i, case in enumerate(cases, 1):
    print(
        f"  [{i:3}/{len(cases)}] "
        f"{case['case_id']} -> {case['filename']}"
    )

case_ids = [c["case_id"] for c in cases]
duplicate_ids = sorted({
    cid for cid in case_ids
    if case_ids.count(cid) > 1
})

if duplicate_ids:
    raise RuntimeError(
        "[ERROR] Duplicate case IDs detected:\n"
        + "\n".join(f"  {cid}" for cid in duplicate_ids)
    )

print(f"\nUnique case IDs verified: {len(case_ids)}")
print("Cell 5 complete.")

In [ ]:
# CELL 6 — Enry language detection, scanner routing & normalized SAST

import csv
import io
import xml.etree.ElementTree as ET


def detect_language_enry(source: str, filename: str) -> str:
    """Detect the language of one source file using the Enry CLI."""
    if not source.strip():
        return "Unknown"

    work = tempfile.mkdtemp(prefix="enry_")
    try:
        path = Path(work) / Path(filename).name
        path.write_text(source, encoding="utf-8")

        r = subprocess.run(
            ["enry", "--json", "--prog", str(path)],
            capture_output=True, text=True, timeout=60
        )
        if r.returncode != 0:
            raise RuntimeError(f"Enry failed: {r.stderr.strip()[:1000]}")

        data = json.loads(r.stdout or "{}")
        # Enry returns {language: [files...]}. Since this temp directory contains
        # exactly one source file, the language containing that filename is the
        # per-file detection result.
        basename = path.name
        for language, files in data.items():
            if basename in files or str(path) in files:
                return language

        # Defensive fallback for CLI versions that return a single language key
        # but omit the path in a one-file invocation.
        if len(data) == 1:
            return next(iter(data))

        return "Unknown"
    finally:
        shutil.rmtree(work, ignore_errors=True)

def route_scanner(language: str) -> str:
    """Map Enry's language name to the scanner used by this pipeline."""
    lang = (language or "").strip().lower()

    if lang in {"python", "python 2", "python 3"}:
        return "bandit"

    if lang == "java":
        return "spotbugs"

    if lang in {
        "c", "c++", "objective-c", "objective-c++",
        "cuda", "cuda c++"
    }:
        return "flawfinder"

    return "semgrep"


def _tmp_source_file(source: str, filename: str):
    suffix = Path(filename).suffix or ".txt"
    f = tempfile.NamedTemporaryFile(
        mode="w", suffix=suffix, delete=False, encoding="utf-8"
    )
    f.write(source)
    f.close()
    return f.name


def _normalize_cwe(value) -> str:
    if value is None:
        return ""
    s = str(value).strip()
    m = re.search(r"CWE[-_: ]?(\d+)", s, re.I)
    return f"CWE-{m.group(1)}" if m else (s if s else "")


def _normalize_finding(cwe, severity, line, description, tool):
    try:
        line_no = int(line)
    except Exception:
        line_no = 0

    return {
        "cwe_id": _normalize_cwe(cwe) or "CWE-N/A",
        "severity": str(severity or "unknown").strip().lower(),
        "line": line_no,
        "description": str(description or "Static-analysis finding").strip(),
        "tool": tool,
    }


def run_flawfinder_scanner(source: str, filename: str) -> list[dict]:
    """Run Flawfinder and parse its CSV output robustly."""
    if not source.strip():
        return []

    tmp = _tmp_source_file(source, filename)
    try:
        r = subprocess.run(
            ["flawfinder", "--csv", "--quiet", tmp],
            capture_output=True, text=True, timeout=60
        )
        if r.returncode not in (0, 1):
            print(f"  [Flawfinder] exit={r.returncode}: {r.stderr[:300]}")

        findings = []
        reader = csv.DictReader(io.StringIO(r.stdout))
        for row in reader:
            raw = " ".join(str(v) for v in row.values() if v)
            cwes = re.findall(r"CWE-(\d+)", raw, re.I)
            cwe = f"CWE-{cwes[0]}" if cwes else ""
            line = row.get("line") or row.get("Line") or "0"
            level = row.get("level") or row.get("Level") or "unknown"
            name = row.get("name") or row.get("Name") or ""
            text = row.get("text") or row.get("Text") or row.get("message") or ""
            description = text or name or row.get("category") or "Flawfinder finding"

            findings.append(
                _normalize_finding(cwe, level, line, description, "flawfinder")
            )
        return findings
    finally:
        try:
            os.unlink(tmp)
        except OSError:
            pass


def run_bandit_scanner(source: str, filename: str) -> list[dict]:
    """Run Bandit against a Python source file and parse JSON."""
    tmp = _tmp_source_file(source, filename)
    try:
        r = subprocess.run(
            ["bandit", "-q", "-f", "json", tmp],
            capture_output=True, text=True, timeout=60
        )
        # Bandit returns non-zero when findings are present.
        if not r.stdout.strip():
            return []

        data = json.loads(r.stdout)
        findings = []
        for item in data.get("results", []):
            issue = item.get("issue_text", "Bandit finding")
            cwe_data = item.get("issue_cwe") or {}
            cwe = cwe_data.get("id", "") if isinstance(cwe_data, dict) else ""
            findings.append(
                _normalize_finding(
                    f"CWE-{cwe}" if cwe else "",
                    item.get("issue_severity", "unknown"),
                    item.get("line_number", 0),
                    issue,
                    "bandit",
                )
            )
        return findings
    finally:
        try:
            os.unlink(tmp)
        except OSError:
            pass


def run_spotbugs_scanner(source: str, filename: str) -> list[dict]:
    """
    Run SpotBugs on a Java source file.

    SpotBugs operates on compiled Java bytecode rather than source text.
    This adapter therefore compiles the temporary source with javac and then
    invokes spotbugs if it is available on PATH.
    """
    spotbugs = shutil.which("spotbugs")
    javac = shutil.which("javac")

    if not spotbugs or not javac:
        raise RuntimeError(
            "SpotBugs routing selected, but 'spotbugs' and/or 'javac' "
            "is not installed. Install SpotBugs and a JDK before processing Java."
        )

    work = tempfile.mkdtemp(prefix="spotbugs_")
    try:
        java_path = Path(work) / Path(filename).name
        java_path.write_text(source, encoding="utf-8")

        classes = Path(work) / "classes"
        classes.mkdir()
        comp = subprocess.run(
            [javac, "-g", "-d", str(classes), str(java_path)],
            capture_output=True, text=True, timeout=60
        )
        if comp.returncode != 0:
            raise RuntimeError(f"javac failed: {comp.stderr[:1000]}")

        xml_path = Path(work) / "spotbugs.xml"
        r = subprocess.run(
            [
                spotbugs, "-textui", "-xml",
                "-output", str(xml_path), str(classes)
            ],
            capture_output=True, text=True, timeout=120
        )
        if not xml_path.exists():
            return []

        root = ET.parse(xml_path).getroot()
        findings = []

        for bug in root.findall(".//BugInstance"):
            bug_type = bug.attrib.get("type", "SpotBugs finding")
            priority = bug.attrib.get("priority", "unknown")
            source_line = bug.find(".//SourceLine")
            line = (
                source_line.attrib.get("start")
                if source_line is not None
                else "0"
            )

            # SpotBugs XML does not always expose a CWE directly. Preserve the
            # scanner message and use N/A when no CWE is supplied.
            cwe = ""
            message = bug_type
            findings.append(
                _normalize_finding(cwe, priority, line, message, "spotbugs")
            )

        return findings
    finally:
        shutil.rmtree(work, ignore_errors=True)


def run_semgrep_scanner(source: str, filename: str) -> list[dict]:
    """Run Semgrep with JSON output and normalize its findings."""
    semgrep = shutil.which("semgrep")
    if not semgrep:
        raise RuntimeError("Semgrep is not installed or not available on PATH.")

    tmp = _tmp_source_file(source, filename)
    try:
        r = subprocess.run(
            [semgrep, "--config", "auto", "--json", "--quiet", tmp],
            capture_output=True, text=True, timeout=120
        )
        if not r.stdout.strip():
            return []

        data = json.loads(r.stdout)
        findings = []

        for item in data.get("results", []):
            extra = item.get("extra", {})
            metadata = extra.get("metadata", {}) or {}
            cwe_values = metadata.get("cwe", "")

            if isinstance(cwe_values, list):
                cwe = cwe_values[0] if cwe_values else ""
            else:
                cwe = cwe_values

            severity = (
                extra.get("severity")
                or metadata.get("severity")
                or "unknown"
            )
            line = (item.get("start") or {}).get("line", 0)
            description = (
                extra.get("message")
                or item.get("check_id")
                or "Semgrep finding"
            )

            findings.append(
                _normalize_finding(
                    cwe, severity, line, description, "semgrep"
                )
            )

        return findings
    finally:
        try:
            os.unlink(tmp)
        except OSError:
            pass


def run_sast_by_language(source: str, filename: str) -> tuple[str, str, list[dict]]:
    """
    Full language-dependent SAST path:
        source -> Enry -> scanner -> normalized findings
    """
    language = detect_language_enry(source, filename)
    scanner = route_scanner(language)

    if scanner == "bandit":
        findings = run_bandit_scanner(source, filename)
    elif scanner == "spotbugs":
        findings = run_spotbugs_scanner(source, filename)
    elif scanner == "flawfinder":
        findings = run_flawfinder_scanner(source, filename)
    else:
        findings = run_semgrep_scanner(source, filename)

    return language, scanner, findings


def format_normalized_findings(findings: list[dict]) -> str:
    """Convert findings to the compact representation used by the L1 prompt."""
    if not findings:
        return "No issues detected by the selected SAST scanner."

    lines = []
    for f in findings:
        lines.append(
            f"[{f['cwe_id']}] | Severity: {f['severity']} | "
            f"Line: {f['line']} | Description: {f['description']}"
        )
    return "\n".join(lines)


def save_sast_record(case: dict, language: str, scanner: str, findings: list[dict]):
    """Persist the actual detection/routing/normalization record for reruns."""
    record_dir = RESULTS_DIR / "sast_records"
    record_dir.mkdir(parents=True, exist_ok=True)

    record = {
        "case_id": case["case_id"],
        "filename": case["filename"],
        "cwe_id": case["cwe_id"],
        "language": language,
        "scanner": scanner,
        "finding_count": len(findings),
        "normalized_findings": findings,
        "normalized_prompt_text": format_normalized_findings(findings),
        "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    }

    path = record_dir / f"{case['case_id']}.json"
    path.write_text(json.dumps(record, indent=2))
    return path


print("Language detection, scanner routing and normalized SAST functions ready.")


In [ ]:
# CELL 7 — L1 prompt builder & LLM client

def build_l1_prompt(source_code: str, normalized_findings: str) -> str:
    """Build the L1 prompt using the normalized SAST representation."""

    return (
        "Role:\n"
        "You are a secure code repair assistant specializing in identifying and "
        "fixing software vulnerabilities in source code.\n\n"

        "Context:\n"

        "Source Code:\n"
        f"{source_code}\n\n"

        "Detected Vulnerabilities (SAST Output):\n"
        f"{normalized_findings}\n\n"

        "Format:\n"
        "[CWE-ID] | Severity: <level> | Line: <number> | Description:\n"
        "<short message>\n\n"

        "Task:\n"
        "Fix the reported vulnerabilities in the code using the provided "
        "diagnostic information.\n\n"

        "Requirements:\n"
        "• Preserve the original functionality of the program.\n"
        "• Modify only the parts of the code necessary to fix the vulnerabilities.\n"
        "• Do not introduce new vulnerabilities or errors.\n"
        "• Ensure the code compiles and runs correctly.\n\n"

        "Output:\n"
        "Return only the fully corrected source code. Do not include explanations, "
        "comments, or additional text."
    )


@torch.inference_mode()
def call_model(prompt: str, max_new_tokens: int = 4096) -> str | None:
    """Single-shot greedy generation, retained from the baseline pipeline."""

    messages = [
        {
            "role": "system",
            "content": "You are a secure code repair assistant."
        },
        {
            "role": "user",
            "content": prompt
        },
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(model.device)

    try:
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=None,
            top_p=None,
            pad_token_id=tokenizer.eos_token_id,
        )

        new_tokens = out[0][inputs["input_ids"].shape[-1]:]

        return tokenizer.decode(
            new_tokens,
            skip_special_tokens=True
        ).strip()

    except Exception as e:
        print(f"  [MODEL ERROR] {e}")
        return None


def extract_code(response: str) -> str:
    """Strip markdown code fences that some models wrap output in."""

    if not response:
        return ""

    fenced = re.search(
        r"```(?:c|cpp|c\+\+)?\n(.*?)```",
        response,
        re.DOTALL
    )

    if fenced:
        return fenced.group(1).strip()

    lines = response.strip().splitlines()

    if lines and lines[0].startswith("```"):
        lines = lines[1:]

    if lines and lines[-1].strip() == "```":
        lines = lines[:-1]

    return "\n".join(lines)


print(
    "Model test:",
    call_model(
        "Say hello in one word.",
        max_new_tokens=10
    )
)

In [ ]:
# CELL 8 — Run L1 generation with Enry routing + normalized SAST

progress_file = RESULTS_DIR / "l1_progress.json"

completed: dict[str, dict] = {}

if progress_file.exists():
    completed = json.loads(
        progress_file.read_text()
    )

    print(
        f"Resuming — {len(completed)} cases already recorded."
    )


print(f"\n{'=' * 70}")
print(
    f"L1 generation: {len(cases)} cases | "
    f"model={MODEL_ID}"
)
print(f"{'=' * 70}\n")


for i, case in enumerate(cases, 1):

    cid = case["case_id"]

    if cid in completed:
        print(
            f"[{i:3}/{len(cases)}] "
            f"SKIP: {cid}"
        )
        continue

    print(
        f"[{i:3}/{len(cases)}] "
        f"CWE-{case['cwe_id']} | "
        f"{case['filename']}"
    )

    case_start = time.time()

    enry_start = time.time()

    try:
        language = detect_language_enry(
            case["source"],
            case["filename"]
        )
    except Exception as e:
        elapsed = round(time.time() - case_start, 3)

        print(
            f"  [ENRY ERROR] {e}"
        )

        completed[cid] = {
            "case_id": cid,
            "cwe_dir": case["cwe_dir"],
            "cwe_id": case["cwe_id"],
            "filename": case["filename"],
            "status": "enry_error",
            "language": "Unknown",
            "scanner": "unknown",
            "enry_error": str(e),
            "elapsed_s": elapsed,
            "fix_len": 0,
            "sast_findings": 0,
        }

        progress_file.write_text(
            json.dumps(completed, indent=2)
        )

        continue

    enry_elapsed = round(
        time.time() - enry_start,
        3
    )

    scanner = route_scanner(language)

    print(
        f"  Enry language: {language}"
    )

    print(
        f"  Selected scanner: {scanner}"
    )

    scanner_start = time.time()

    try:

        if scanner == "bandit":

            findings = run_bandit_scanner(
                case["source"],
                case["filename"]
            )

        elif scanner == "spotbugs":

            findings = run_spotbugs_scanner(
                case["source"],
                case["filename"]
            )

        elif scanner == "flawfinder":

            findings = run_flawfinder_scanner(
                case["source"],
                case["filename"]
            )

        else:

            findings = run_semgrep_scanner(
                case["source"],
                case["filename"]
            )

    except Exception as e:

        scanner_elapsed = round(
            time.time() - scanner_start,
            3
        )

        case_elapsed = round(
            time.time() - case_start,
            3
        )

        print(
            f"  [SAST ERROR] {e}"
        )

        completed[cid] = {
            "case_id": cid,
            "cwe_dir": case["cwe_dir"],
            "cwe_id": case["cwe_id"],
            "filename": case["filename"],
            "status": "sast_error",
            "language": language,
            "scanner": scanner,
            "sast_error": str(e),
            "enry_elapsed_s": enry_elapsed,
            "scanner_elapsed_s": scanner_elapsed,
            "elapsed_s": case_elapsed,
            "fix_len": 0,
            "sast_findings": 0,
        }

        progress_file.write_text(
            json.dumps(completed, indent=2)
        )

        continue

    scanner_elapsed = round(
        time.time() - scanner_start,
        3
    )

    print(
        f"  Findings: {len(findings)}"
    )

    normalized = format_normalized_findings(
        findings
    )

    record_path = save_sast_record(
        case,
        language,
        scanner,
        findings
    )

    print(
        f"  Normalized record: {record_path}"
    )

    prompt = build_l1_prompt(
        case["source"],
        normalized
    )

    model_start = time.time()

    raw = call_model(
        prompt,
        max_new_tokens=4096
    )

    model_elapsed = round(
        time.time() - model_start,
        3
    )

    fix = extract_code(
        raw
    ) if raw else ""

    status = (
        "generated"
        if raw
        else "model_error"
    )

    out_file = (
        OUTPUT_DIR
        / case["cwe_dir"]
        / case["filename"]
    )

    out_file.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    out_file.write_text(
        fix,
        encoding="utf-8"
    )

    case_elapsed = round(
        time.time() - case_start,
        3
    )

    completed[cid] = {
        "case_id": cid,
        "cwe_dir": case["cwe_dir"],
        "cwe_id": case["cwe_id"],
        "filename": case["filename"],

        "status": status,

        "language": language,
        "scanner": scanner,

        "enry_elapsed_s": enry_elapsed,
        "scanner_elapsed_s": scanner_elapsed,
        "model_elapsed_s": model_elapsed,
        "elapsed_s": case_elapsed,

        "sast_findings": len(findings),
        "normalized_sast": normalized,
        "sast_record": str(record_path),

        "fix_len": len(fix),
        "output_file": str(out_file),
    }

    progress_file.write_text(
        json.dumps(
            completed,
            indent=2
        )
    )

    print(
        f"  -> [{status}] "
        f"Enry={enry_elapsed}s | "
        f"SAST={scanner_elapsed}s | "
        f"Qwen={model_elapsed}s | "
        f"Total={case_elapsed}s | "
        f"Fix={len(fix)} chars"
    )

    print()


print(
    f"\n{'=' * 70}"
)

print(
    f"Generation complete: "
    f"{len(completed)}/{len(cases)} cases recorded."
)

print(
    f"Progress file: {progress_file}"
)

print(
    f"{'=' * 70}"
)

In [ ]:
# CELL 9 — Evaluation helpers

def _compile_and_run(
    code: str, filename: str, tmpdir: str, label: str = "test"
) -> dict:
    """
    Compile a single Juliet file with ASan+UBSan, link against io.c, and run it.
    Returns: {compiles, runs, compile_err, run_err}
    """
    ext = ".cpp" if filename.endswith(".cpp") else ".c"
    cc  = "g++" if ext == ".cpp" else "gcc"

    for hfile in HARNESS_DIR.iterdir():
        if hfile.is_file():
            shutil.copy2(str(hfile), os.path.join(tmpdir, hfile.name))

    io_src   = os.path.join(tmpdir, "io.c")
    src_path = os.path.join(tmpdir, f"{label}_{filename}")
    bin_path = os.path.join(tmpdir, f"{label}_bin")

    with open(src_path, "w") as f:
        f.write(code)

    compile_cmd = [
        cc, "-DINCLUDEMAIN",
        "-fsanitize=address,undefined", "-fno-sanitize-recover=all",
        "-I", tmpdir, src_path, io_src, "-o", bin_path,
        "-lm", "-lpthread",
    ]
    try:
        comp = subprocess.run(compile_cmd, capture_output=True, text=True, timeout=30)
    except subprocess.TimeoutExpired:
        return {"compiles": False, "runs": False, "compile_err": "Compile timed out", "run_err": ""}
    except Exception as e:
        return {"compiles": False, "runs": False, "compile_err": str(e), "run_err": ""}

    if comp.returncode != 0:
        err = comp.stderr[:1500]
        if "undefined reference to `main'" in comp.stderr:
            err = "Model removed #ifdef INCLUDEMAIN block (no main() compiled in)"
        return {"compiles": False, "runs": False, "compile_err": err, "run_err": ""}

    env = {**os.environ, "ASAN_OPTIONS": "detect_leaks=0", "UBSAN_OPTIONS": "print_stacktrace=1"}
    for run_idx in range(NUM_HARNESS_RUNS):
        try:
            run = subprocess.run(
                [bin_path], capture_output=True, text=True, timeout=10, env=env,
            )
        except subprocess.TimeoutExpired:
            return {"compiles": True, "runs": False, "compile_err": "",
                    "run_err": f"Runtime timed out (run {run_idx+1}/{NUM_HARNESS_RUNS})"}
        except Exception as e:
            return {"compiles": True, "runs": False, "compile_err": "", "run_err": str(e)}
        if run.returncode != 0:
            err = run.stderr[:500] if run.stderr else f"Exit code {run.returncode}"
            return {"compiles": True, "runs": False, "compile_err": "", "run_err": err}

    return {"compiles": True, "runs": True, "compile_err": "", "run_err": ""}


def juliet_harness_test(original: str, fix: str, filename: str, cwe_id: str) -> dict:
    """
    Compile + run the original (baseline crash check) and the fix (clean exit check).
    Semantic CWEs (SAST_ONLY_CWES) skip the runtime test — falls through to Flawfinder only.
    """
    if not fix.strip():
        return {
            "baseline_crashes": False, "harness_reliable": False,
            "fix_compiles": False, "fix_runs": False,
            "compile_err": "Empty source", "run_err": "",
        }

    runtime_meaningful = cwe_id not in SAST_ONLY_CWES

    with tempfile.TemporaryDirectory() as tmpdir:
        baseline_crashes = False
        if runtime_meaningful:
            orig = _compile_and_run(original, filename, tmpdir, label="orig")
            baseline_crashes = orig["compiles"] and not orig["runs"]

        fix_result = _compile_and_run(fix, filename, tmpdir, label="fix")
        return {
            "baseline_crashes": baseline_crashes,
            "harness_reliable": runtime_meaningful,
            "fix_compiles":     fix_result["compiles"],
            "fix_runs":         fix_result["runs"] if runtime_meaningful else fix_result["compiles"],
            "compile_err":      fix_result["compile_err"],
            "run_err":          fix_result["run_err"],
        }


def run_flawfinder(code: str, filename: str) -> list[dict]:
    """Run Flawfinder on a code string, return list of findings with CWE IDs."""
    if not code.strip():
        return []
    ext = ".cpp" if filename.endswith(".cpp") else ".c"
    with tempfile.NamedTemporaryFile(mode="w", suffix=ext, delete=False) as f:
        f.write(code)
        tmp = f.name
    try:
        r = subprocess.run(
            ["flawfinder", "--csv", "--quiet", tmp],
            capture_output=True, text=True, timeout=60,
        )
        findings = []
        for line in r.stdout.strip().splitlines()[1:]:
            parts = line.split(",")
            if len(parts) >= 5:
                cwe_nums = re.findall(r"CWE-(\d+)", parts[-1])
                findings.append({
                    "line":     parts[1].strip().strip('"'),
                    "level":    parts[3].strip().strip('"'),
                    "category": parts[4].strip().strip('"') if len(parts) > 4 else "",
                    "name":     parts[5].strip().strip('"') if len(parts) > 5 else "",
                    "cwes":     cwe_nums,
                    "raw":      line,
                })
        return findings
    except Exception as e:
        print(f"  [Flawfinder error] {e}")
        return []
    finally:
        os.unlink(tmp)


def function_deleted(original: str, fix: str) -> bool:
    """Heuristic: fix < 30% the size of original → function body was gutted."""
    return len(fix.strip()) < len(original.strip()) * 0.30

In [ ]:
# CELL 10 — Classification logic (CFR / PIR / IFR)

def classify(original: str, fix: str, filename: str, known_cwe: str) -> dict:
    """
    Classify a fix:
      IFR pre-checks  → empty / identical / gutted
      Stage 1 harness → compile fail / runtime crash (memory-safety CWEs only)
      Stage 2 SAST    → CFR if known CWE gone + no new CWEs; PIR otherwise
    """
    if not fix.strip():
        return {"outcome": "IFR", "reason": "Empty output",
                "compile_ok": False, "runs_ok": False, "harness_reliable": False}
    if fix.strip() == original.strip():
        return {"outcome": "IFR", "reason": "Identical to original",
                "compile_ok": False, "runs_ok": False, "harness_reliable": False}
    if function_deleted(original, fix):
        return {"outcome": "IFR", "reason": "Function body deleted (< 30% original size)",
                "compile_ok": False, "runs_ok": False, "harness_reliable": False}

    harness = juliet_harness_test(original, fix, filename, known_cwe)

    if not harness["fix_compiles"]:
        return {
            "outcome": "IFR",
            "reason": f"Compile failed: {harness['compile_err'][:200]}",
            "compile_ok": False, "runs_ok": False,
            "harness_reliable": harness["harness_reliable"],
        }
    if harness["harness_reliable"] and not harness["fix_runs"]:
        return {
            "outcome": "IFR",
            "reason": f"Runtime crash (ASan/UBSan): {harness['run_err'][:200]}",
            "compile_ok": True, "runs_ok": False, "harness_reliable": True,
            "baseline_crashes": harness["baseline_crashes"],
        }

    orig_findings = run_flawfinder(original, filename)
    fix_findings  = run_flawfinder(fix, filename)

    orig_cwes = {c for f in orig_findings for c in f["cwes"]}
    fix_cwes  = {c for f in fix_findings  for c in f["cwes"]}

    known_resolved = (known_cwe not in fix_cwes) if (known_cwe in orig_cwes) else True
    new_cwes = fix_cwes - orig_cwes

    if known_resolved and not new_cwes:
        outcome, reason = "CFR", "Harness pass + known CWE resolved + no new CWEs"
    elif new_cwes:
        outcome, reason = "PIR", f"New CWEs introduced: {sorted(new_cwes)}"
    else:
        outcome, reason = "PIR", f"Known CWE-{known_cwe} still flagged by Flawfinder"

    return {
        "outcome":             outcome,
        "reason":              reason,
        "compile_ok":          True,
        "runs_ok":             harness["fix_runs"],
        "harness_reliable":    harness["harness_reliable"],
        "baseline_crashes":    harness.get("baseline_crashes", False),
        "known_cwe":           known_cwe,
        "known_resolved":      known_resolved,
        "orig_cwes":           sorted(orig_cwes),
        "fix_cwes":            sorted(fix_cwes),
        "new_cwes":            sorted(new_cwes),
        "orig_findings_count": len(orig_findings),
        "fix_findings_count":  len(fix_findings),
    }

In [ ]:
# CELL 11 — Run L1 evaluation

results = []
counts  = {"CFR": 0, "PIR": 0, "IFR": 0}

print(f"Evaluating {len(cases)} L1 fixes...\n")

for i, case in enumerate(cases, 1):
    fix_path = OUTPUT_DIR / case["cwe_dir"] / case["filename"]

    if not fix_path.exists():
        cl = {"outcome": "IFR", "reason": "No output file generated",
              "compile_ok": False, "runs_ok": False}
    else:
        fix_src = fix_path.read_text(errors="replace")
        cl = classify(case["source"], fix_src, case["filename"], case["cwe_id"])

    counts[cl["outcome"]] += 1
    results.append({
        "case_id":  case["case_id"],
        "cwe_dir":  case["cwe_dir"],
        "cwe_id":   case["cwe_id"],
        "filename": case["filename"],
        **cl,
    })
    print(f"[{i:3}/{len(cases)}] [{cl['outcome']}] {case['case_id']}  —  {cl['reason']}")

total = len(results)
print(f"\n{'='*60}")
print(f"L1 Results — {total} cases | model={MODEL_ID}")
print(f"{'='*60}")
print(f"  CFR (Correct Fix Rate):         {counts['CFR']:3}  ({100*counts['CFR']/total:.1f}%)")
print(f"  PIR (Plausible but Incorrect):  {counts['PIR']:3}  ({100*counts['PIR']/total:.1f}%)")
print(f"  IFR (Invalid Fix Rate):         {counts['IFR']:3}  ({100*counts['IFR']/total:.1f}%)")

In [ ]:
# CELL 12 — Save L1 results + per-CWE breakdown

final = {
    "level":       "L1",
    "model":       MODEL_ID,
    "prompt_type": "raw_code_plus_normalized_sast",
    "timestamp":   datetime.now(timezone.utc).isoformat(),
    "total_cases": total,
    "summary": {
        "CFR":     counts["CFR"],
        "PIR":     counts["PIR"],
        "IFR":     counts["IFR"],
        "CFR_pct": round(100 * counts["CFR"] / total, 1),
        "PIR_pct": round(100 * counts["PIR"] / total, 1),
        "IFR_pct": round(100 * counts["IFR"] / total, 1),
    },
    "cases": results,
}

results_file = RESULTS_DIR / "l1_results.json"
results_file.write_text(json.dumps(final, indent=2, default=str))
print(f"Results saved -> {results_file}")

print("\nPer-CWE Breakdown:")
print(f"{'CWE Directory':<50} {'N':>3} {'CFR':>4} {'PIR':>4} {'IFR':>4}")
print("-" * 70)
for cwe in sorted(set(r["cwe_dir"] for r in results)):
    cwe_r = [r for r in results if r["cwe_dir"] == cwe]
    n   = len(cwe_r)
    cfr = sum(1 for r in cwe_r if r["outcome"] == "CFR")
    pir = sum(1 for r in cwe_r if r["outcome"] == "PIR")
    ifr = sum(1 for r in cwe_r if r["outcome"] == "IFR")
    print(f"  {cwe:<48} {n:>3} {cfr:>4} {pir:>4} {ifr:>4}")

l2_pool = [r for r in results if r["outcome"] != "CFR"]
l2_file = RESULTS_DIR / "l2_failure_pool.json"
l2_file.write_text(json.dumps(l2_pool, indent=2, default=str))
print(f"\nL2 failure pool: {len(l2_pool)} cases  ->  {l2_file}")